
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Building AI Agent Tools with Unity Catalog Functions

## Overview 
In this hands-on lab, you will build AI agent tools using Unity Catalog functions. You'll implement both Python and SQL functions, learn to identify and fix inadequate function descriptions, and test your tools using AI Playground.

## Learning Objectives
_By the end of this lab, you will be able to:_
- Create and register a SQL function using SQL syntax with proper documentation
- Build and register a Python function in Unity Catalog using `DatabricksFunctionClient()`
- Identify and fix inadequate function descriptions for AI agent use cases
- Test both functions independently using multiple methods
- Validate both functions as agent tools using AI Playground

### Business Context
You work for a transportation analytics company that wants to provide AI-powered insights about NYC taxi trips. Your team needs to build reliable, governed tools that agents can use to perform trip analysis and fare calculations. These tools must be accurate, well-documented, and accessible through Unity Catalog for governance and security.

## A. Environment Setup

### A1. Compute Requirements

**🚨 REQUIRED - SELECT SERVERLESS COMPUTE**

This course has been configured to run on Serverless compute. While classic compute may also work, testing has been performed on serverless.

**This demo was tested using version 5 of Serverless compute.** To ensure that you are using the correct version of Serverless, please [see this documentation on viewing and changing your notebook's Serverless verison.](https://docs.databricks.com/aws/en/compute/serverless/dependencies) 

### A2. Install Dependencies
As part of the workspace setup, several Python libraries have been installed. To see the list of notebook-scoped libraries, please read [this documentation](https://docs.databricks.com/aws/en/compute/serverless/dependencies#configure-environment-for-job-tasks). In particular, we installed:

1. `unitycatalog-ai[databricks]`: This package provides infrastructure and tooling for creating and managing UC functions (both SQL and Python UDFs) that can be used as tools by agents.

This demonstration uses AI Playground to test our functions, which provides a no-code interface for prototyping tool-calling agents. See the [Unity Catalog Tool Integration documentation](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unity-catalog-tool-integration) for more details for advanced framework integration.

In [0]:
!pip install unitycatalog-ai[databricks]==0.3.2 reportlab
%restart_python

In [0]:
%run ../Includes/Classroom-Setup-1.3

### A3. Inspect NYC Taxi Dataset

The dataset that will be used for this lab will come from the table `samples.nyctaxi.trips`, which is a sample dataset that [is available by default on UC-enabled workspaces](https://docs.databricks.com/aws/en/discover/databricks-datasets). Run the next cell to query the first few rows of the dataset. 

In [0]:
df = spark.read.table('samples.nyctaxi.trips')
display(df.limit(5))

### A4. Initialize the Databricks Function Client

**TODO:** Initialize the DatabricksFunctionClient which provides a programmatic interface for creating, managing, and executing Unity Catalog functions.

In [0]:
## Import and initialize the DatabricksFunctionClient for serverless compute
from unitycatalog.ai.core.databricks import <FILL_IN>

client = <FILL_IN>

##### Task A4: Initialize the Databricks Function Client ANSWER
<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient(execution_mode="serverless")
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

## B. Build a SQL Function for Average Trip Distance

First, let's drop any existing functions with the same name as what will be created below.

In [0]:
%sql
DROP FUNCTION IF EXISTS avg_fare_by_zip;
DROP FUNCTION IF EXISTS est_taxi_fare;

### B1. Create the SQL Function

**TODO:** Create a SQL function that calculates the average trip distance for trips originating from a specific pickup zip code using the `samples.nyctaxi.trips` table.

**Requirements:**
- Function name: `avg_fare_by_zip`
- Parameter: `pickup_zip_code` (INT)
- Return type: DOUBLE
- Include proper COMMENT clauses for both the function and parameter
- Mark as DETERMINISTIC
- Handle NULL values appropriately
- Query the `samples.nyctaxi.trips` table

In [0]:
%sql
-- TODO
-- -- Create the SQL function using CREATE OR REPLACE FUNCTION
CREATE OR REPLACE FUNCTION <FILL_IN>(
  pickup_zip_code <FILL_IN> COMMENT "<FILL_IN>"
)
RETURNS <FILL_IN>
LANGUAGE <FILL_IN>
DETERMINISTIC
COMMENT '<FILL_IN>'
RETURN 
  SELECT <FILL_IN>
  FROM samples.nyctaxi.trips
  WHERE <FILL_IN>
    AND <FILL_IN>

##### Task B1: Create the SQL Function ANSWER
<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
CREATE OR REPLACE FUNCTION avg_fare_by_zip(
  pickup_zip_code INT COMMENT "The pickup zip code to filter trips by (e.g., 10001 for Midtown Manhattan)"
)
RETURNS DOUBLE
LANGUAGE SQL
DETERMINISTIC
COMMENT 'Calculates the average trip distance in miles for all NYC taxi trips originating from a specific pickup zip code. Returns the average distance as a numeric value.'
RETURN 
  SELECT AVG(trip_distance)
  FROM samples.nyctaxi.trips
  WHERE pickup_zip = pickup_zip_code
    AND trip_distance IS NOT NULL
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### B2. Test the SQL Function

**TODO:** Test your SQL function using direct SQL queries to verify it works correctly. Test with zip code 10001 (Midtown Manhattan).

In [0]:
%sql
-- TODO
-- -- Test the function with zip code 10001
SELECT <FILL_IN>(<FILL_IN>) AS avg_distance

##### Task B2: Test the SQL Function ANSWER
<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
SELECT avg_fare_by_zip(10001) AS avg_distance
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

## C. Build a Python Function for Fare Estimation

### C1. A Bad Python Example

**TODO:** Test the following Python function in AI Playground. The original intention of this function is to estimate the total fare for a taxi trip based on distance and time. Run the next cell to bypass best practices and immediately push the code as a UC tool and open the AI Playground for testing.

As you can see from the function's definition, the fare calculation formula is:
- Base fare: $3.00
- Per mile: $2.50
- Per minute: $0.50
- Total = base_fare + (distance * per_mile_rate) + (time_minutes * per_minute_rate)

In [0]:
def est_taxi_fare(
    my_param: float, 
    param2: float
) -> float:
    """
    This is my calculation function.
    
    Args:
        my_param: parameter_1
        param2: my second parameter for my function
    
    Returns:
        the answer
    """
    base_fare = 3.00
    per_mile_rate = 2.50
    per_minute_rate = 0.50
    
    total_fare = base_fare + (my_param * per_mile_rate) + (param2 * per_minute_rate)
    return total_fare

function_info = client.create_python_function(
    func=est_taxi_fare,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
)

print("Python function registered successfully!")
print(f"Function name: {function_info.full_name}")

### C2. Test the Function
Navigate to **AI Playground** and attach the tool `est_taxi_fare()`:

To test your Python functions as agent tools in AI Playground:

1. Navigate to **Playground** from your Databricks workspace
1. Select a model with the **Tools enabled** label (e.g., `GPT OSS 120B` or another tool-capable model) from the model selection dropdown menu at the top of the Playground
1. Click **Use endpoint** to begin your session
1. Click **Tools > + Add tool**
1. Under **UC Function**, select **Hosted Function** as the tool type
1. Select the function you created: `est_taxi_fare`
1. Click **Save** at the bottom right
1. Validate that the tool is equipped. You should see **Tools (1)** in the **Tools** dropdown menu
1. Enter the following question and observe that the model will _not_ use the tool as intended. Instead, you will see the model attempt to reason its way to the answer _without_ the tool call.

> I'm traveling next week and I think it's about 20 minutes and 7 miles. How much will it cost me?

Note that the answer should be calculated as
- Base fare: $3.00
- Per mile: $2.50
- Per minute: $0.50
Total = 3 + (7 * 2.5) + (20 * 0.5) = **30.5 dollars**
However, the LLM will confuse the first input with the second input (because of the incomplete lack of context brought on by the docstring) and return a value of **56.50 dollars**.

## D. Identify and Fix the Inadequate Function Description

### D1. Understanding the Problem

The Python function you created has an inadequate description. For example, the agent won't be able to determine what the function actually does, what the parameters represent, or what the return value means based on the current docstring. Let's fix that.

**TODO:** Review your function's docstring above. Identify what's missing or unclear.

### D2. Fix the Python Function Description

**TODO:** Redefine the Python function with a comprehensive, detailed docstring that follows best practices for AI agent tools.

**Best practices for function descriptions:**
1. Clear, concise summary of what the function does
2. Detailed parameter descriptions including units and examples
3. Explanation of the return value
4. Context about when to use this function
5. Any important notes or limitations

In [0]:
## Redefine the function with an improved docstring
def est_taxi_fare(
    distance_miles: float, 
    time_minutes: float
) -> float:
    """
    TODO: Write a comprehensive docstring that includes:
    - Clear description of what the function does
    - Detailed parameter descriptions with units and examples
    - Clear return value description
    - Context about the fare calculation method
    
    Args:
        distance_miles: <FILL_IN>
        time_minutes: <FILL_IN>
    
    Returns:
        <FILL_IN>
    """
    base_fare = 3.00
    per_mile_rate = 2.50
    per_minute_rate = 0.50
    
    total_fare = base_fare + (distance_miles * per_mile_rate) + (time_minutes * per_minute_rate)
    return total_fare

##### Task D2: Fix the Python Function Description ANSWER
<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
def est_taxi_fare(
    distance_miles: float, 
    time_minutes: float
) -> float:
    """
    Estimates the total fare for a NYC taxi trip based on distance and duration.
    
    This function calculates the estimated fare using NYC taxi rate structure:
    - Base fare: $3.00
    - Distance rate: $2.50 per mile
    - Time rate: $0.50 per minute
    
    Use this function to provide fare estimates before a trip or to validate fare calculations.
    
    Args:
        distance_miles (float): The trip distance in miles (e.g., 5.5 for a 5.5-mile trip). Must be non-negative.
        time_minutes (float): The trip duration in minutes (e.g., 15.0 for a 15-minute trip). Must be non-negative.
    
    Returns:
        float: The estimated total fare in US dollars. For example, a 5-mile trip taking 15 minutes 
               would return 18.50 (3.00 + 12.50 + 7.50).
    """
    base_fare = 3.00
    per_mile_rate = 2.50
    per_minute_rate = 0.50
    
    total_fare = base_fare + (distance_miles * per_mile_rate) + (time_minutes * per_minute_rate)
    return total_fare
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### D3. Test the Function
Following best practices, let's test the new function here in the notebook before registering with Unity Catalog.
**TODO**: Run the next cell to test a distance of **7 miles** and a time of **20 minutes**, giving a total of **30.5** that we calculated earlier.

In [0]:
est_taxi_fare(
        distance_miles= 7.0,
        time_minutes= 20.0
        )

### D4. Re-register the Python Function with Improved Description
Now we need to re-register our bad Python example. This can be accomplished with the `create_python_function()` API.
> We can wrap the Python function in SQL code and use `CREATE OR REPLACE FUNCTION`.

**TODO:** Re-register the Python function with the improved description. Be sure to set `replace=True` to overwrite the original version of the function.

In [0]:
## Re-register the function with the improved docstring
function_info_improved = client.<FILL_IN>(
    func=<FILL_IN>,
    catalog=<FILL_IN>,
    schema=<FILL_IN>,
    replace=<FILL_IN>
)

print("Python function re-registered with improved description!")
print(f"Function name: {function_info_improved.full_name}")

##### Task D4: Re-register the Python Function with Improved Description ANSWER
<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
function_info_improved = client.create_python_function(
    func=est_taxi_fare,
    catalog=catalog_name,
    schema=schema_name,
    replace=True
)

print("Python function re-registered with improved description!")
print(f"Function name: {function_info_improved.full_name}")
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### D5. Verify the Improved Function Still Works
Next, let's test our new version of the function.

**TODO:** Test the re-registered function to ensure it still works correctly. Note, you should get the same result as testing the function directly from the function definition above.

In [0]:
## Test the improved function
result_improved = client.<FILL_IN>(
    function_name=f"{<FILL_IN>}.{<FILL_IN>}.est_taxi_fare",
    parameters={
        "<FILL_IN>": <FILL_IN>,
        "<FILL_IN>": <FILL_IN>
    }
)

print(f"Estimated Fare (Improved Function): ${result_improved.value}")

##### Task D5: Verify the Improved Function Still Works ANSWER
<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
result_improved = client.execute_function(
    function_name=f"{catalog_name}.{schema_name}.est_taxi_fare",
    parameters={
        "distance_miles": 7.0,
        "time_minutes": 20.0
    }
)

print(f"Estimated Fare (Improved Function): ${result_improved.value}")
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

## E. Testing Tools with AI Playground

### E1. Attach your Tools

To test your functions as agent tools in AI Playground:

1. Navigate to **Playground** from your Databricks workspace
1. Select a model with the **Tools enabled** label (e.g., `GPT OSS 120B`) from the model selection dropdown menu at the top of the **Playground**
1. Click **Use endpoint**


Now let's attach both functions you created as tools for the AI agent:

1. Click **Tools > + Add tool**
2. Under **UC Function**, click on **Hosted Function** as the tool type
3. Select `avg_fare_by_zip` from your catalog and schema
4. Click **Save** at the bottom right
5. Repeat steps 1-4 to add the `est_taxi_fare` function
6. Validate that both tools are equipped; you should see **Tools (2)** in the **Tools** dropdown menu

### E2. Test your Tools

#### Checking what the agent has access to
Before sending actual queries for the use case, you can check to see what your agent has access to with something like 
> What tools can you use?
or 
> What tools do you have access to?

You will see a response that lists all the tools available (it should list the two we attached).

#### Begin sending questions

Now you're ready to send some sample questions and begin testing your functions. Here are some sample questions to help you get started:

> Sample Question 1: I'm traveling next week and I think it's about 20 minutes and 7 miles. How much will it cost me?
- Note this is the same question we used in section **C2**.

> Sample Question 2: What is the average trip distance for pickups from zip code 10001?
- This will test the SQL function

> Can you get the average trip distance in miles for zip code 10001? I want to see how much it would take to travel for 20 mins for that zip code.
- This will (hopefully) call both functions. Keep in mind that outputs vary depending on the type of LLM being leveraged. Below is a screenshot of an example output using **GPT OSS 20B**. If one (or both) functions are not called, you can add a system prompt explicitly telling the LLM to call available functions by clicking on **Add system prompt** in the Playground and sending a message like _"Exhaust all tool options before deriving your answer"_.

<!-- Output image -->

![optional alt text](../Includes/images/llm_output.png)

## Conclusion
The skills you've developed enable you to create production-ready agent tools that combine Unity Catalog's governance framework with the power of both SQL and Python. By following best practices for function descriptions, you ensure that AI agents can effectively understand and utilize your tools to provide accurate, reliable insights. 

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>